In [ ]:
# Install required packages from PyPI.
!pip install unsloth bitsandbytes peft

import torch
from unsloth import FastLanguageModel
from peft import PeftModel

# Set some hyperparameters
max_seq_length = 2048  # Unsloth supports any value; it scales RoPE internally.
load_in_4bit = True    # Use 4-bit quantization to reduce memory usage.
dtype = None           # Auto-detects the optimal data type.

# ---------------------------------------------------------------------------
# 1. Load the Base Model Using Unsloth's FastLanguageModel API
# ---------------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    full_finetuning=False,  # Set to False for adapter finetuning
    dtype=dtype,
)

# ---------------------------------------------------------------------------
# 2. Apply LoRA Adapter Configuration Using Unsloth's get_peft_model
#    This patches the model to support efficient LoRA finetuning.
# ---------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Optimized for efficiency; adjust if needed.
    bias="none",
    use_gradient_checkpointing="unsloth",  # Optimizes VRAM usage.
    random_state=3407,
    max_seq_length=max_seq_length,
    use_rslora=False,
    loftq_config=None,
)

# ---------------------------------------------------------------------------
# 3. Load Pre-trained LoRA Weights from the Hugging Face Hub
#    (Adapter model: "millat/StudyAbroadGPT-7B-LoRa-Kaggle")
# ---------------------------------------------------------------------------
model = PeftModel.from_pretrained(model, "millat/StudyAbroadGPT-7B-LoRa-Kaggle")

# ---------------------------------------------------------------------------
# 4. Prepare the Model for Inference
#    This enables Unsloth’s 2x faster native inference optimizations.
# ---------------------------------------------------------------------------
FastLanguageModel.for_inference(model)
model.eval()

In [16]:
# ---------------------------------------------------------------------------
# 5. Run a Prompt Through the Model
# ---------------------------------------------------------------------------
prompt = "Given my background as a third-year Computer Science & Engineering student at Sharda University with an expected CGPA of 6/10, could you outline the post-graduation career pathways in Italy, including job market prospects and any available support for international graduates in this field?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=20000,  # Adjust as needed
        do_sample=True,
        temperature=0.7
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)


Given my background as a third-year Computer Science & Engineering student at Sharda University with an expected CGPA of 6/10, could you outline the post-graduation career pathways in Italy, including job market prospects and any available support for international graduates in this field?

Italy is known for its rich history, culture, and architecture, but it also has a thriving technology sector, particularly in fields like Computer Science & Engineering. Here's an outline of potential post-graduation career pathways in Italy for an international student in your field:

1. **Software Developer/Engineer**: This is one of the most common roles for computer science graduates. In Italy, you can find positions in various sectors, including tech companies, startups, and multinational corporations. Salaries can vary depending on the company, location, and your experience level. According to Glassdoor, the average salary for a Software Developer in Italy is around €35,000 per year.

2. **Dat